# Construir Gold — KPIs de negócio e observabilidade

Constrói as tabelas Gold a partir da Silver: KPIs de negócio (reconciliação financeira, OTIF, qualidade de produção) e observabilidade (falha cruzada de cadeia fria, qualidade de SKU, estoque negativo).

Diferente da Silver (MERGE por chave única, incremental), a Gold é recriada por completo a cada execução (`overwrite`) — é resultado derivado da Silver, não fonte de verdade própria.

Referências: business-context.md (KPIs documentados), ADR-011 (falha cruzada de cadeia fria).

In [0]:
# gold_reconciliacao_financeira
from pyspark.sql.functions import col, abs as spark_abs

df_pedidos = spark.table("poc_pulse_observability.silver.crm_pedidos")
df_faturas = spark.table("poc_pulse_observability.silver.financeiro_faturas")

df_reconciliacao = (
    df_faturas
    .join(df_pedidos.select("pedido_id", "valor_total"), "pedido_id")
    .withColumn("divergencia_valor", col("valor_faturado") - col("valor_total"))
    .withColumn("divergente", spark_abs(col("divergencia_valor")) > 0.01)
    .select(
        "fatura_id", "pedido_id", "valor_total", "valor_faturado",
        "divergencia_valor", "divergente", "data_faturamento",
    )
)

df_reconciliacao.write.format("delta").mode("overwrite").saveAsTable(
    "poc_pulse_observability.gold.gold_reconciliacao_financeira"
)

total = df_reconciliacao.count()
divergentes = df_reconciliacao.filter(col("divergente")).count()
print(f"Total: {total} | Divergentes: {divergentes} ({divergentes/total:.1%})")

In [0]:
# gold_otif
from pyspark.sql.functions import col, when

df_remessas = spark.table("poc_pulse_observability.silver.tms_remessas")
df_comprovantes = spark.table("poc_pulse_observability.silver.tms_comprovantes_entrega")

df_otif = (
    df_comprovantes
    .join(df_remessas.select("remessa_id", "data_entrega_prevista"), "remessa_id")
    .withColumn(
        "no_prazo",
        when(col("pod_confirmado") == True, col("data_entrega_real") <= col("data_entrega_prevista")).otherwise(None)
    )
    .select(
        "remessa_id", "comprovante_id", "data_entrega_prevista", "data_entrega_real",
        "pod_confirmado", "no_prazo", "status_entrega",
    )
)

df_otif.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.gold.gold_otif")

total = df_otif.count()
confirmadas = df_otif.filter(col("pod_confirmado") == True).count()
no_prazo = df_otif.filter(col("no_prazo") == True).count()
print(f"Total: {total} | Confirmadas: {confirmadas} | No prazo: {no_prazo} ({no_prazo/confirmadas:.1%} das confirmadas)")

In [0]:
# gold_qualidade_producao
from pyspark.sql.functions import col, count, sum as spark_sum, when

df_lotes = spark.table("poc_pulse_observability.silver.erp_lotes_producao")

df_qualidade = (
    df_lotes
    .groupBy("centro_producao_id", "produto_id")
    .agg(
        count("*").alias("total_lotes"),
        spark_sum(when(col("status_qc") == "reprovado", 1).otherwise(0)).alias("lotes_reprovados"),
    )
    .withColumn("taxa_rejeicao", col("lotes_reprovados") / col("total_lotes"))
)

df_qualidade.write.format("delta").mode("overwrite").saveAsTable(
    "poc_pulse_observability.gold.gold_qualidade_producao"
)

df_qualidade.orderBy(col("taxa_rejeicao").desc()).show(20, truncate=False)

## Fase B — Observabilidade de qualidade e dados

Regras cruzadas entre sistemas, que só a plataforma de observabilidade consegue detectar (nenhum sistema de origem, sozinho, veria o problema).

In [0]:
# observability_cadeia_fria - Passo 1 : LEFT JOIN preservando as 789 remessas
from pyspark.sql.functions import col, when, lit

df_remessas = spark.table("poc_pulse_observability.silver.tms_remessas")
df_notas = spark.table("poc_pulse_observability.silver.erp_notas_expedicao").select("nota_expedicao_id", "lote_id")
df_lotes = spark.table("poc_pulse_observability.silver.erp_lotes_producao").select("lote_id", "produto_id")
df_produtos = spark.table("poc_pulse_observability.silver.erp_produtos").select("produto_id", "nome_produto", "exige_cadeia_fria")
df_veiculos = spark.table("poc_pulse_observability.silver.tms_veiculos").select("veiculo_id", "refrigerado")

df_base = (
    df_remessas.select("remessa_id", "nota_expedicao_id", "veiculo_id")
    .join(df_notas, "nota_expedicao_id", "left")
    .join(df_lotes, "lote_id", "left")
    .join(df_produtos, "produto_id", "left")
    .join(df_veiculos, "veiculo_id", "left")
    .withColumn(
        "verificavel",
        col("produto_id").isNotNull() & col("refrigerado").isNotNull()
    )
)

total = df_base.count()
verificaveis = df_base.filter(col("verificavel")).count()
nao_verificaveis = total - verificaveis
print(f"Total: {total} | Verificáveis: {verificaveis} | Não verificáveis: {nao_verificaveis} ({nao_verificaveis/total:.1%})")

In [0]:
# observability_cadeia_fria - Passo 2: leitura de temperatura (para veículo refrigerado)
df_leituras = spark.table("poc_pulse_observability.silver.tms_leituras_temperatura")

df_temperatura_fora_faixa = (
    df_leituras
    .filter((col("temperatura_celsius") < 2.0) | (col("temperatura_celsius") > 8.0))
    .select("remessa_id")
    .distinct()
    .withColumn("teve_leitura_fora_faixa", lit(True))
)

df_com_temperatura = df_base.join(df_temperatura_fora_faixa, "remessa_id", "left")
df_com_temperatura = df_com_temperatura.fillna({"teve_leitura_fora_faixa": False})

df_com_temperatura.filter(col("verificavel")).groupBy("teve_leitura_fora_faixa").count().show()

In [0]:
# observability_cadeia_fria - Passo 3: classificação final e gravação
from pyspark.sql.functions import when

df_classificado = (
    df_com_temperatura
    .withColumn(
        "tipo_violacao",
        when(~col("verificavel"), "nao_verificavel")
        .when(~col("exige_cadeia_fria"), "nao_aplicavel")
        .when(~col("refrigerado"), "veiculo_incorreto")
        .when(col("teve_leitura_fora_faixa"), "falha_equipamento")
        .otherwise("conforme")
    )
    .select(
        "remessa_id", "veiculo_id", "produto_id", "nome_produto",
        "exige_cadeia_fria", "refrigerado", "teve_leitura_fora_faixa", "tipo_violacao",
    )
)

df_classificado.write.format("delta").mode("overwrite").saveAsTable(
    "poc_pulse_observability.observability.observability_cadeia_fria"
)

df_classificado.groupBy("tipo_violacao").count().orderBy(col("count").desc()).show()

### observability_qualidade_sku

Detecta itens de pedido referenciando produto_id inexistente no catálogo (docs/schemas/crm.md — SKU inexistente, 3% de propósito). Mais simples que a cadeia fria: só 2 tabelas, sem dependência em cascata.

In [0]:
# observability_qualidade_sku
from pyspark.sql.functions import col

df_itens = spark.table("poc_pulse_observability.silver.crm_itens_pedido")
df_produtos_validos = spark.table("poc_pulse_observability.silver.erp_produtos").select(
    col("produto_id").alias("produto_id_valido")
)

df_sku = (
    df_itens
    .join(df_produtos_validos, df_itens.produto_id == col("produto_id_valido"), "left")
    .withColumn("sku_valido", col("produto_id_valido").isNotNull())
    .select("item_pedido_id", "pedido_id", "produto_id", "sku_valido")
)

df_sku.write.format("delta").mode("overwrite").saveAsTable(
    "poc_pulse_observability.observability.observability_qualidade_sku"
)

total = df_sku.count()
invalidos = df_sku.filter(~col("sku_valido")).count()
print(f"Total: {total} | SKU inválido: {invalidos} ({invalidos/total:.1%})")

### observability_estoque_negativo

Detecta posições de estoque com quantidade negativa (docs/schemas/erp.md — 5% de propósito). A mais simples das 3 tabelas de observabilidade: 1 tabela só, sem JOIN.

In [0]:
# observability_estoque_negativo
df_estoque = spark.table("poc_pulse_observability.silver.erp_posicoes_estoque")

df_estoque_negativo = df_estoque.select(
    "posicao_id", "lote_id", "centro_distribuicao_id", "quantidade", "data_posicao"
).withColumn("estoque_invalido", col("quantidade") < 0)

df_estoque_negativo.write.format("delta").mode("overwrite").saveAsTable(
    "poc_pulse_observability.observability.observability_estoque_negativo"
)

total = df_estoque_negativo.count()
negativos = df_estoque_negativo.filter(col("estoque_invalido")).count()
print(f"Total: {total} | Estoque negativo: {negativos} ({negativos/total:.1%})")